# 02 - Modélisation : rétention des professionnels de santé

Prédire si un HCP engagé en année N le sera encore en N+1, expliquer le modèle
(SHAP), segmenter les profils d'engagement, et vérifier l'équité entre spécialités.
Nécessite `data/openpayments.sqlite` (notebook 01).

In [ ]:
import sqlite3, pathlib
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM_FEATURES = ["n_payments", "total_amount", "mean_amount", "n_manufacturers", "n_natures",
                "share_food", "share_travel", "share_consulting", "share_speaker", "share_education"]
CAT_FEATURES = ["specialty"]
TARGET = "retenu"

ROOT = pathlib.Path.cwd(); ROOT = ROOT if (ROOT / "data").exists() else ROOT.parent
DB_PATH = ROOT / "data" / "openpayments.sqlite"

with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql("SELECT * FROM hcp_features", con)
print("Profils charges:", len(df))

## Modèle de rétention et évaluation (vs baseline)

In [ ]:
def build_pipeline():
    pre = ColumnTransformer([
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_FEATURES),
    ])
    clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                 class_weight="balanced", random_state=0, n_jobs=-1)
    return Pipeline([("pre", pre), ("clf", clf)])

X = df[NUM_FEATURES + CAT_FEATURES]
y = df[TARGET].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

base = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
pipe = build_pipeline().fit(X_tr, y_tr)
proba = pipe.predict_proba(X_te)[:, 1]

print("Taux de retention   :", round(y.mean(), 3))
print("ROC-AUC baseline    :", round(roc_auc_score(y_te, base.predict_proba(X_te)[:, 1]), 3))
print("ROC-AUC RandomForest:", round(roc_auc_score(y_te, proba), 3))
print("PR-AUC  RandomForest:", round(average_precision_score(y_te, proba), 3))
print(classification_report(y_te, (proba >= 0.5).astype(int), digits=3))

## Explicabilité : importances et SHAP

In [ ]:
names = pipe.named_steps["pre"].get_feature_names_out()
imp = pd.Series(pipe.named_steps["clf"].feature_importances_, index=names).sort_values(ascending=False)
print("Importances (top 10):")
print(imp.head(10).round(3).to_string())

try:
    import shap
    Xs = df[NUM_FEATURES + CAT_FEATURES].sample(min(500, len(df)), random_state=0)
    Xt = pipe.named_steps["pre"].transform(Xs)
    vals = shap.TreeExplainer(pipe.named_steps["clf"]).shap_values(Xt)
    arr = vals[1] if isinstance(vals, list) else np.asarray(vals)
    if arr.ndim == 3:
        arr = arr[:, :, 1]
    top = pd.Series(np.abs(arr).mean(axis=0), index=names).sort_values(ascending=False).head(10)
    print("\nSHAP (contributions moyennes):")
    print(top.round(4).to_string())
except ImportError:
    print("(SHAP non installe : pip install shap)")

## Segmentation des profils d'engagement (k-means)

In [ ]:
Xseg = StandardScaler().fit_transform(df[NUM_FEATURES].fillna(0))
d = df.copy()
d["segment"] = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(Xseg)
seg = d.groupby("segment").agg(
    n_hcp=("segment", "size"),
    total_amount_moyen=("total_amount", "mean"),
    n_payments_moyen=("n_payments", "mean"),
    n_labos_moyen=("n_manufacturers", "mean"),
    taux_retention=(TARGET, "mean"),
).round(2)
seg

## Équité (fairness) : performance par spécialité

In [ ]:
proba_cv = cross_val_predict(build_pipeline(), X, y, cv=5, method="predict_proba", n_jobs=-1)[:, 1]
d2 = df.copy(); d2["proba"] = proba_cv
print(f"  {'specialite':45s}  {'n':>5s}  {'retention':>9s}  {'AUC':>6s}")
for sp in d2["specialty"].value_counts().head(6).index:
    m = d2["specialty"] == sp
    if m.sum() >= 50 and d2.loc[m, TARGET].nunique() > 1:
        auc = roc_auc_score(d2.loc[m, TARGET], d2.loc[m, "proba"])
        print(f"  {str(sp)[:45]:45s}  {m.sum():5d}  {d2.loc[m, TARGET].mean():9.2f}  {auc:6.3f}")